# TerraScout AI — Producto 1: ETL Geoespacial y EDA Real

Notebook de análisis alineado con [`actors/apify_etl/my_actor/etl_core.py`](../actors/apify_etl/my_actor/etl_core.py) y [`actors/apify_eda/my_actor/eda_core.py`](../actors/apify_eda/my_actor/eda_core.py):
- **Cero datos sintéticos:** 100% fuentes reales en `data/raw/` y `data/aux_topografia_cordoba.csv`.
- **Reglas desacopladas:** lectura dinámica desde `config/business_rules.json`.
- **Trazabilidad:** embudo ETL (`1.200 -> 894`), correlaciones reales (Pearson, Spearman, Kendall e Información Mutua) y voz del turista (`1.754` reseñas Google Maps).

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path('.').resolve()
if not (REPO_ROOT / 'config' / 'business_rules.json').exists() and (REPO_ROOT.parent / 'config' / 'business_rules.json').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from actors.apify_etl.my_actor.etl_core import run_etl_pipeline
from actors.apify_eda.my_actor.eda_core import run_eda_pipeline

df_clean, funnel = run_etl_pipeline()
display(pd.DataFrame(funnel))
print(f'Dataset limpio oficial: {len(df_clean)} lotes x {df_clean.shape[1]} columnas | Oportunidades P1: {int(df_clean["es_oportunidad_p1"].sum())}')

In [ ]:
clean_path = REPO_ROOT / 'data' / 'processed' / 'dataset_limpio.csv'
clean_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(clean_path, index=False, encoding='utf-8')

eda_res = run_eda_pipeline(dataset_limpio_source=str(clean_path))
print(json.dumps(eda_res['summary'], indent=2, ensure_ascii=False))

print('\n--- Hoja 1: Resumen por Localidad y Riesgo ---')
display(eda_res['csv_1_localidades_y_riesgo'].head(10))

print('\n--- Hoja 2: Correlaciones Reales (Pearson, Spearman, Kendall, Info Mutua) ---')
df_c2 = eda_res['csv_2_correlaciones_y_distribuciones']
display(df_c2[df_c2['bloque'] == 'correlacion_features'])